# 04. RAG Pipeline End-to-End Evaluation

Complete evaluation of the RAG pipeline with retrieval and generation metrics.


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from config.settings import RAGConfig
from src.rag_pipeline import NutritionRAG
from src.evaluation import RAGEvaluator, AnswerQualityEvaluator
import time

/Users/soham/Desktop/nutribot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data


In [2]:
df = pd.read_csv('../data/processed/nutrition_docs.csv')
documents = df['content'].tolist()
print(f"Loaded {len(documents)} documents")

Loaded 10 documents


## Initialize RAG Pipeline


In [3]:
config = RAGConfig()
rag = NutritionRAG(config)
rag.setup_hybrid_retriever(documents)
print("RAG pipeline initialized!")

2026-06-15 20:26:27,345 - src.rag_pipeline - INFO - Initializing NutritionRAG pipeline
2026-06-15 20:26:27,345 - src.embeddings.factory - INFO - Creating minilm embedder
2026-06-15 20:26:27,346 - src.embeddings.minilm - INFO - Loading MiniLM embedder from sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22748.32it/s]
2026-06-15 20:26:29,881 - src.embeddings.minilm - INFO - MiniLM embedder loaded successfully on cpu
2026-06-15 20:26:29,882 - src.rag_pipeline - INFO - Embedder initialized: sentence-transformers/all-MiniLM-L6-v2 (384d)
2026-06-15 20:26:29,882 - src.rag_pipeline - INFO - Using hybrid retriever (BM25 + semantic)
2026-06-15 20:26:30,354 - src.rag_pipeline - ERROR - Failed to initialize Groq LLM: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable
2026-06-15 20:26:30,354 - src.rag_pipeline - INFO - RAG pipeline initialized successfully
2026-06-15 20:26:3

RAG pipeline initialized!


## Test Queries with Ground Truth


In [4]:
test_queries = [
    {'query': 'How much protein do I need daily?', 'relevant_docs': [0]},
    {'query': 'What are good sources of healthy fats?', 'relevant_docs': [2]},
    {'query': 'How much water should I drink?', 'relevant_docs': [4]},
]
print(f"Testing with {len(test_queries)} queries")

Testing with 3 queries


## Run RAG Pipeline


In [5]:
all_results = []

for test in test_queries:
    query = test['query']
    relevant_docs = test['relevant_docs']
    print(f"\\nQuery: {query}")
    print("="*60)
    start = time.time()
    result = rag.query(query, prompt_type='standard', top_k=5)
    elapsed = time.time() - start
    retrieved_docs = result['retrieved_docs']
    retrieved_scores = result['scores']
    retrieval_metrics = RAGEvaluator.evaluate(
        retrieved_docs, retrieved_scores, 
        [documents[i] for i in relevant_docs], k=5
    )
    print(f"Retrieval Metrics:")
    for metric, value in retrieval_metrics.items():
        print(f"  {metric}: {value:.3f}")
    print(f"Latency: {elapsed*1000:.1f}ms")
    all_results.append({'query': query, 'latency_ms': elapsed*1000, **retrieval_metrics})

results_df = pd.DataFrame(all_results)
print("\\n" + "="*60)
print("Summary Statistics:")
print(results_df.describe())

2026-06-15 20:26:34,955 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:34,956 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:26:34,957 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:34,957 - src.rag_pipeline - INFO - Query completed in 0.01s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:26:34,965 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:34,965 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:26:34,966 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:34,966 - src.rag_pipeline - INFO - Query completed in 0.01s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:26:34,974 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:34,974 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:26:34,975 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:34,975 - sr

\nQuery: How much protein do I need daily?
Retrieval Metrics:
  mrr: 1.000
  ndcg@5: 1.000
  ndcg@10: 1.000
  precision@5: 0.200
  precision@10: 0.100
  recall@5: 1.000
  recall@10: 1.000
Latency: 15.1ms
\nQuery: What are good sources of healthy fats?
Retrieval Metrics:
  mrr: 1.000
  ndcg@5: 1.000
  ndcg@10: 1.000
  precision@5: 0.200
  precision@10: 0.100
  recall@5: 1.000
  recall@10: 1.000
Latency: 8.5ms
\nQuery: How much water should I drink?
Retrieval Metrics:
  mrr: 1.000
  ndcg@5: 1.000
  ndcg@10: 1.000
  precision@5: 0.200
  precision@10: 0.100
  recall@5: 1.000
  recall@10: 1.000
Latency: 9.0ms
\n============================================================
Summary Statistics:
       latency_ms  mrr  ndcg@5  ndcg@10   precision@5  precision@10  recall@5  \
count    3.000000  3.0     3.0      3.0  3.000000e+00  3.000000e+00       3.0   
mean    10.878086  1.0     1.0      1.0  2.000000e-01  1.000000e-01       1.0   
std      3.704242  0.0     0.0      0.0  3.399350e-17  1.69967

## Answer Quality Evaluation


In [6]:
quality_eval = AnswerQualityEvaluator()

for test in test_queries:
    query = test['query']
    result = rag.query(query, prompt_type='standard')
    contexts = result['retrieved_docs']
    answer = result['response']
    relevance = quality_eval.evaluate_answer_relevance(query, answer)
    faithfulness = quality_eval.evaluate_faithfulness('\\n'.join(contexts), answer)
    precision = quality_eval.evaluate_context_precision(contexts, answer)
    print(f"\\nQuery: {query}")
    print(f"  Answer Relevance: {relevance:.3f}")
    print(f"  Faithfulness: {faithfulness:.3f}")
    print(f"  Context Precision: {precision:.3f}")

2026-06-15 20:26:41,660 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:41,660 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:26:41,661 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:41,661 - src.rag_pipeline - INFO - Query completed in 0.01s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:26:41,667 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:41,667 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:26:41,668 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:41,668 - src.rag_pipeline - INFO - Query completed in 0.01s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:26:41,675 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:26:41,675 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:26:41,676 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:26:41,676 - sr

\nQuery: How much protein do I need daily?
  Answer Relevance: 0.000
  Faithfulness: 0.000
  Context Precision: 0.000
\nQuery: What are good sources of healthy fats?
  Answer Relevance: 0.000
  Faithfulness: 0.000
  Context Precision: 0.000
\nQuery: How much water should I drink?
  Answer Relevance: 0.000
  Faithfulness: 0.000
  Context Precision: 0.000
